# ShopDesk, Section 1 Lab 2: A Second Tool and a System Prompt

A beginner-friendly notebook built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. We take the single-tool agent
from Lab 1, add a **second tool** and a **system prompt** (role and goal), and watch how
the prompt and the tool descriptions change which tool the agent picks. This is the last
step before the full agentic loop.

## The real-world scenario

One tool answered status questions. But ShopDesk also handles refunds, so the agent now has
to **choose** between two tools, and it needs a **role and a goal** so it chooses well. Two
things steer that choice: the **system prompt** (who the agent is and what it must do) and
the **tool descriptions** (what each tool is for).

The question this lab answers: **once an agent has more than one tool, how do the prompt and
the tool descriptions decide what it does?**

## Objectives

- Add a **second tool** so the agent must select, not just call the only option.
- Add a **system prompt** that gives the agent a role and a goal.
- Observe how changing the **system prompt** and a **tool description** changes the agent's
  decision on the same request.
- See why a request that needs two tools *in order* points to the **agentic loop** next.

## What you'll observe

- A shipping question routes to the status tool; a refund question routes to the refund tool.
- Strengthening the system prompt ("check status before refunding") makes the agent look up
  the order first instead of refunding blindly.
- A request that needs two steps completes only its first tool call in a single round-trip,
  which is exactly what the loop will fix.

## How to run

Run top to bottom. The tools, schemas, and system-prompt cells are pure Python and run
anywhere. The experiment cells call Claude, so paste a real key into **Setup 2/3** and
re-run from the top; otherwise they skip and the notes describe what you would see. Live
model choices are not perfectly deterministic, so treat each run as an observation.

## 0. Setup

**This cell:** installs the packages. Same base SDK as Lab 1; adding a tool and a
system prompt does not change the plumbing, only the inputs to the call.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets the `RUN_LIVE` switch so
live calls fire only with a real key. Same setup as Lab 1, kept here so this notebook stands
on its own.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import json                                     # print tool payloads readably
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: two orders, their statuses, and
whether each is refundable. A1 is shipped and refundable; A2 was delivered and is past the
30-day window. The tools read from this.

In [ ]:
# ===== SETUP 3/3 - the shared ShopDesk data =====
ORDERS = {                                       # our tiny order book (the agent's environment)
    "A1": {"status": 2, "refundable": True},     #   shipped,   within the window
    "A2": {"status": 3, "refundable": False},    #   delivered, past the window
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> human word
print("orders:", list(ORDERS), "| refund window: 30 days")   # quick sanity check

---

### 🎯 Lab objective - two tools, a role, and what steers the choice

**What you build:** a two-tool ShopDesk agent with a system prompt, plus small experiments
that change the prompt and a tool description and show the decision moving.

**Why it helps you build real solutions:** in any real agent, *what it does* is controlled
by the prompt and the tool descriptions far more than by clever code. Learning to steer with
those is the core skill.

**How you'll see it:** the same request routes differently as you edit the system prompt or a
tool's description, and a two-step request reveals why you need a loop.

**This cell:** the **two tool bodies**. `get_order_status` reads the status (from Lab
1); `refund_order` is the new second tool, applying the 30-day rule. Both are plain Python;
the model only asks us to run them.

In [ ]:
# ===== the two tool bodies (plain Python) =====
def get_order_status(order_id):                  # tool 1: look up a status
    o = ORDERS.get(order_id)                     #   find the order
    return STATUS_NAMES[o["status"]] if o else "unknown order"   # status word, or a miss

def refund_order(order_id):                      # tool 2: attempt a refund
    o = ORDERS.get(order_id)                     #   find the order
    if not o:                                    #   no such order?
        return "unknown order"                   #     say so
    if not o["refundable"]:                      #   past the window?
        return "refused: past 30-day window"     #     the business rule
    return "refunded"                            #   otherwise success

RUN_TOOL = {"get_order_status": get_order_status,   # name -> function, so we can dispatch by name
            "refund_order": refund_order}
print("tool bodies ready:", list(RUN_TOOL))      # confirm both exist

**This cell:** the **two schemas** the model reads. The `description` on each is the
routing signal: it is how the model tells a shipping question from a refund one. We write
both descriptions sharply now; later we will blur one to see the effect.

In [ ]:
# ===== the two tool schemas (sharp descriptions) =====
_arg = {"type": "object",                         # both tools take one string order_id
        "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"]}
TOOLS = [                                          # the tool list handed to the model
    {"name": "get_order_status",
     "description": "Look up the delivery status of an order. Use for tracking and shipping questions.",
     "input_schema": _arg},
    {"name": "refund_order",
     "description": "Refund an order if it is within the 30-day window. Use for refund and money-back requests.",
     "input_schema": _arg},
]
print("tools:", [t["name"] for t in TOOLS])       # confirm both are registered

**This cell:** the **system prompt**, the prompt building block Lab 1 left out. It
gives the agent a role, a tone, and a goal. This is separate from the tools; it shapes *how*
the agent behaves across every request.

In [ ]:
# ===== the system prompt: role and goal =====
SYSTEM = (                                         # who the agent is and what it should do
    "You are ShopDesk, a calm, concise e-commerce support agent. "
    "Your goal is to resolve the customer's request using your tools. "
    "Use get_order_status for shipping questions and refund_order for refund requests."
)
print(SYSTEM)                                      # show the role and goal we will send

**This cell:** a **round-trip helper** like Lab 1's, now taking a `system` and a
`tools` list, and it reports the tool the agent chose. It handles one round-trip; if the
model wants a *second* tool afterward, it says so, which is the hook into the loop.

In [ ]:
# ===== one round-trip, but now the agent must CHOOSE a tool =====
def run_once(question, system, tools):            # returns the final text; prints the choice
    client = anthropic.Anthropic()                #   the LLM client
    messages = [{"role": "user", "content": question}]   # the goal

    first = client.messages.create(model=MODEL, max_tokens=512,   # the model decides
                                   system=system, tools=tools, messages=messages)
    if first.stop_reason != "tool_use":           #   chose not to use a tool?
        return "".join(b.text for b in first.content if b.type == "text")   # just answer

    call = next(b for b in first.content if b.type == "tool_use")   # the chosen tool
    print("  chosen tool:", call.name, call.input)                  #   observe the decision
    result = RUN_TOOL[call.name](**call.input)     # run it
    messages.append({"role": "assistant", "content": first.content})       # keep the model's turn
    messages.append({"role": "user", "content": [   # hand the result back
        {"type": "tool_result", "tool_use_id": call.id, "content": result}]})

    second = client.messages.create(model=MODEL, max_tokens=512,   # let the model continue
                                    system=system, tools=tools, messages=messages)
    if second.stop_reason == "tool_use":           #   it wants ANOTHER tool
        nxt = next(b for b in second.content if b.type == "tool_use")
        print("  still wants:", nxt.name, "-> a LOOP would continue here (next section)")
        return "(incomplete: needs another step)"  #   one round-trip cannot finish this
    return "".join(b.text for b in second.content if b.type == "text")   # the final answer

**This cell:** experiment 1, **routing by intent**. We send a shipping question and a
refund question through the same agent and watch it pick a different tool for each. The tool
descriptions are doing the routing.

In [ ]:
# ===== experiment 1: the same agent routes two intents =====
if RUN_LIVE:                                       # needs a real key
    print("Q: shipping ->"); print("A:", run_once("Where is order A1 right now?", SYSTEM, TOOLS))
    print()
    print("Q: refund ->");   print("A:", run_once("I want a refund on order A2.", SYSTEM, TOOLS))
else:
    print("[skipped] expected: the shipping question picks get_order_status,")
    print("          the refund question picks refund_order.")

**This cell:** experiment 2, **the system prompt changes the decision**. We swap in a
stricter prompt that tells the agent to check the order's status *before* refunding. On the
refund request the agent now reaches for `get_order_status` first. Same tools, same request:
only the prompt changed.

In [ ]:
# ===== experiment 2: a stricter system prompt changes what the agent does =====
SYSTEM_STRICT = (                                  # same role, plus a new must-do rule
    "You are ShopDesk, a calm, concise e-commerce support agent. "
    "Before you ever refund an order, you MUST first check its status with get_order_status. "
    "Never call refund_order without checking status first."
)
if RUN_LIVE:                                       # needs a real key
    print("with the strict prompt, the refund request now starts by checking status:")
    print("A:", run_once("I want a refund on order A2.", SYSTEM_STRICT, TOOLS))
else:
    print("[skipped] expected: the agent now picks get_order_status FIRST, then reports")
    print("          that a second step (the refund) still remains -> that needs a loop.")

**This cell:** experiment 3, **the tool description changes the decision**. We blur the
refund tool's description so it no longer says what it is for, then send a refund request. A
vague description makes the agent hesitate or mis-route, which shows how much the description
carries the routing.

In [ ]:
# ===== experiment 3: a vague tool description degrades routing =====
TOOLS_VAGUE = [                                    # same tools, but the refund description is blurred
    {"name": "get_order_status",
     "description": "Look up the delivery status of an order. Use for tracking and shipping questions.",
     "input_schema": _arg},
    {"name": "refund_order",
     "description": "Does refund stuff.",          #   vague: little for the model to route on
     "input_schema": _arg},
]
if RUN_LIVE:                                       # needs a real key
    print("with a vague refund description, the same request may route poorly:")
    print("A:", run_once("I want a refund on order A2.", SYSTEM, TOOLS_VAGUE))
else:
    print("[skipped] expected: routing is less reliable; the sharp description in TOOLS")
    print("          is what made experiment 1 pick the refund tool confidently.")

**This cell:** the **bridge to the loop**. A single request that needs two tools in
order (verify, then refund) cannot finish in one round-trip: the helper handles the first
tool and then reports that another is still wanted. That leftover step is exactly what the
agentic loop in the next section will carry to completion.

In [ ]:
# ===== a two-step request shows why we need a loop =====
if RUN_LIVE:                                       # needs a real key
    print("A:", run_once("Check order A2's status, then refund it.", SYSTEM_STRICT, TOOLS))
else:
    print("[skipped] expected: one round-trip does step 1 (status) and then says another")
    print("          tool is still wanted. The loop (next section) keeps going until done.")

| anti-pattern | what to do instead |
|---|---|
| leave tool descriptions vague and hope | write a sharp description that says exactly when to use each tool |
| bury a rule in code the model cannot see | put must-do rules in the system prompt where they steer the choice |
| assume one round-trip finishes any task | recognise multi-step requests and let a loop continue them |
| tune behaviour by rewriting the round-trip code | tune the prompt and the descriptions first; they carry most of the behaviour |

**Lesson:** with more than one tool, an agent's behaviour is steered mostly by two
inputs you control: the **system prompt** (its role and rules) and the **tool descriptions**
(what each tool is for). Change either and the decision moves. And once a task needs several
tools in order, one round-trip is not enough, which is the reason the next section builds the
**loop**.

---

## Recap - what steers an agent

| Lever | What we changed | What moved |
|---|---|---|
| Second tool | added `refund_order` | the agent now selects instead of calling the only option |
| System prompt | added a role and a must-do rule | the agent checked status before refunding |
| Tool description | blurred the refund description | routing became less reliable |
| Multi-step request | asked for status then refund | one round-trip stalled -> motivates the loop |

One principle to carry forward: **steer the agent with the prompt and the tool descriptions;
reach for the loop when one round-trip cannot finish the job.** To run live, paste a real key
into **Setup 2/3** and re-run from the top. Then try it: add a third tool, or reword a
description, and watch the routing shift. Next section turns the single round-trip into the
full agentic loop.